This it the collab notebook to implement a RAG.

The inputs or source/knowledge base are the pdfs in the folder /RAG/documents.

The pdfs can be text or images.

The vector db is Chroma db under /RAG/db.

The hosting is using gradio.

The RAG pipeline is separated into a data ingestion, that populates the db.

The embeddings that creates the embeddings.

The retrieval that retrieves the data from db.

The  generation that generates the output.

## Load the Enviornment files that have the keys



In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
#
groq_key = os.getenv("GROQ_KEY")
hugging_face_key = os.getenv("HUGGING_FACE_KEY")

if groq_key.startswith("gsk_"):
  print(f"Groq Key loaded!")
else:
  print(f"Failed to load Groq Key!")

if hugging_face_key.startswith("hf_"):
  print(f"Hugging Key loaded!")
else:
  print(f"Failed to load Hugging Face Key!")

Groq Key loaded!
Hugging Key loaded!


## The Data ingestion

In [4]:
!pip install -q langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is 

In [5]:
from langchain_community.document_loaders import pyPDFLoader
import os
def load_documents(knowledge_base_path:str):
  documents = []
  for file in os.listdir(knowledge_base_path):
    file_path = os.path.join(knowledge_base_path, file)
    loader = PyPDFLoader(file_path)
    doc = loader.load()
    documents.extend(doc)
    print(f"Loaded {len(doc)} pages from {file_path}")
    return documents
  print(f"Loaded total {len(documents)} documents")


ImportError: cannot import name 'pyPDFLoader' from 'langchain_community.document_loaders' (/usr/local/lib/python3.12/dist-packages/langchain_community/document_loaders/__init__.py)

## Create Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunkizer(documents:list, chunk_size:int, chunk_overlap:int):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )
  chunks = text_splitter.split_documents(documents)
  print(f"Chunizer: num documents: {len(documents}, Chunk Sz: {chunk_size} Chunk OverLap: {chunk_overlap}")
  print(f"Chunkizer created {lem(chunks)} chunks")
  return chunks

## Storing the emeddings in vector DB

In [ ]:
import shutil
import os
from langchain.vectorstores import Chroma

vector_store = None
def add_to_vector_db(vector_db_path:str, chunks:list, embedding_model ,purge: True):
  if purge:
    if os.path.exists(vector_db_path):
      print(f"Purging {vector_db_path}")
      shutil.rmtree(vector_db_path
      if os.path.exists(vector_db_path):
        print(f"Purging Failed: {vector_db_path}")
      else:
        print(f"Purged {vector_db_path}")

  embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
  if not os.exists(vector_db_path):
    vector_store = Chroma.from_documents(documents = chunks, embedding = embedding_model, persist_directory = vector_db_path)
  else:
    vector_store.upsert()

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings


## Method to do similarity search

In [ ]:
def vector_db_similarity_search(vector_db, search_str:):
  results = vector_db.similarity_search("mmr", serach_str)
  print(f"Similiarity search returned {len(results)} chunk\")
  return resultsvec

## Use LLM to generate text

In [ ]:
from groq import Grpq
def generate_response(system_role: str, query: str)
  groq_client = Groq(api_key =groq_key)
  try:
    result = groq_client.chat.completions.create(
        model = "llama-3.1-8b-instant",
        messages = [
            {
                "role":"system",
                "content": system_role,
            },
            {
                "role":"user"
                "content":query
            }
        ]
    )
    return result.choices[0].message.content
  except Exception as e:
    print(f"Groq Connection failed: {e}")



## Rag Pipeline